# **Citi Bike Data Engineering  - EDA - Value Distribution** 

#### Python Package

In [12]:
import pandas as pd
import numpy as np
import csv
import sys
import os

#### Python Package

In [2]:
# Dynamically add the project root to sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from scripts.create_markdown_table import create_markdown_table

#### Import DataFrames

In [3]:
df_dir = os.path.join(project_root, 'eda', 'dataframes')
newark_airport_df = pd.read_pickle(os.path.join(df_dir, 'newark_airport_df.pkl'))
citibike_df = pd.read_pickle(os.path.join(df_dir, 'citibike_df.pkl'))

# Confirm that the dataframes are loaded correctly
if newark_airport_df.empty or citibike_df.empty:
    raise ValueError("One or both dataframes are empty. Please check the data loading process.")
else:
    print("Dataframes loaded successfully.")

Dataframes loaded successfully.


#### Identify low-caridinality columns for dimension tables

In [4]:
# Create a dictionary for business context related to wind speeds
wind_speed_bounds_data = [
    {"Description": "Calm to Light Breeze", "min_wind_speed": 0, "max_wind_speed": 10, "Notes": "Common daily conditions"},
    {"Description": "Moderate to Strong Breeze", "min_wind_speed": 10, "max_wind_speed": 40, "Notes": "Typical during gusty days or storms"},
    {"Description": "Gale to Severe Gale", "min_wind_speed": 40, "max_wind_speed": 74, "Notes": "Rare but possible in strong weather systems"},
    {"Description": "Hurricane-force Winds", "min_wind_speed": 75, "max_wind_speed": 100, "Notes": "Extreme, often coastal or storm-driven"}
]
wind_speed_df = pd.DataFrame(wind_speed_bounds_data)

In [5]:
# Create a dictionary for business context related to temperature bounds in New Jersey
nj_temperature_bounds_data = [
    {"Description": "Frigid Winter Lows", "min_temp_f": -20, "max_temp_f": 20, "Notes": "Rare but possible in inland or northern NJ during deep winter"},
    {"Description": "Typical Winter Range", "min_temp_f": 21, "max_temp_f": 40, "Notes": "Common from December to February"},
    {"Description": "Mild Spring/Fall", "min_temp_f": 41, "max_temp_f": 65, "Notes": "Transitional seasons, typical March–May and October–November"},
    {"Description": "Warm Summer Days", "min_temp_f": 66, "max_temp_f": 85, "Notes": "Typical June–August daytime highs"},
    {"Description": "Extreme Summer Heat", "min_temp_f": 86, "max_temp_f": 105, "Notes": "Occasional heatwaves, especially in urban areas"}
    ]

nj_temp_df = pd.DataFrame(nj_temperature_bounds_data)

In [9]:
# Create a list of dataframes for the following steps
dfs = {"Newark Airport Weather Data": newark_airport_df, "Citi Bike Data": citibike_df
}

In [11]:
def identify_low_cardinality_tables(dfs, threshold=0.05):
    """
    Identify low-cardinality columns in each DataFrame, create dimension tables,
    and return metadata to support future ETL workflows.

    Args:
        dfs (dict): Dictionary of {table_name: DataFrame}
        threshold (float): Max ratio of unique values / total rows

    Returns:
        dict: {
            table_name: {
                'dimension_tables': {col_name: dimension_df},
                'efficiency_scale': float,
                'etl_metadata': {
                    'fk_mappings': {col_name: fk_column_name},
                    'transforms': [dict],
                    'ddl_snippets': [str]
                },
                'cardinality_detail': [(col_name, ratio)]
            }
        }
    """
    result = {}

    for name, df in dfs.items():
        dim_tables = {}
        cardinality_metrics = []
        fk_mappings = {}
        transforms = []
        ddl_snippets = []

        for col in df.columns:
            unique_vals = df[col].nunique(dropna=False)
            cardinality_ratio = unique_vals / len(df)

            if cardinality_ratio <= threshold:
                # Generate dimension table
                values = pd.Series(df[col].dropna().unique()).reset_index(drop=True)
                dim_df = pd.DataFrame({
                    f"{col}_id": range(1, len(values)+1),
                    col: values
                })

                dim_tables[col] = dim_df
                fk_col = f"{col}_id"
                fk_mappings[col] = fk_col
                cardinality_metrics.append((col, cardinality_ratio))

                # Add ETL-friendly transform and DDL note
                transforms.append({
                    'source_column': col,
                    'transform': f"Map to {fk_col} via lookup on {col}_dim"
                })
                ddl_snippets.append(
                    f"-- Dimension table for {col}\n"
                    f"CREATE TABLE {col}_dim (\n"
                    f"    {fk_col} SERIAL PRIMARY KEY,\n"
                    f"    {col} TEXT\n);"
                )

        efficiency = len(dim_tables) / len(df.columns) if df.columns.size else 0

        result[name] = {
            'dimension_tables': dim_tables,
            'efficiency_scale': round(efficiency, 2),
            'cardinality_detail': cardinality_metrics,
            'etl_metadata': {
                'fk_mappings': fk_mappings,
                'transforms': transforms,
                'ddl_snippets': ddl_snippets
            }
        }

    return result

# Identify low cardinality tables and generate metadata
low_cardinality_results = identify_low_cardinality_tables(dfs)

low_cardinality_results

{'Newark Airport Weather Data': {'dimension_tables': {'STATION':    STATION_id      STATION
   0           1  USW00014734,
   'NAME':    NAME_id                                         NAME
   0        1  NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US,
   'PGTM': Empty DataFrame
   Columns: [PGTM_id, PGTM]
   Index: [],
   'SNOW':     SNOW_id  SNOW
   0         1   0.0
   1         2   0.7
   2         3   0.5
   3         4   0.3
   4         5  24.0
   5         6   0.2
   6         7   2.8
   7         8   1.5
   8         9   1.4
   9        10   0.1
   10       11   3.0,
   'SNWD':     SNWD_id  SNWD
   0         1   0.0
   1         2   1.2
   2         3   7.1
   3         4  20.1
   4         5  18.9
   5         6  16.9
   6         7  14.2
   7         8   9.8
   8         9   9.1
   9        10   7.9
   10       11   3.9
   11       12   2.0,
   'TSUN': Empty DataFrame
   Columns: [TSUN_id, TSUN]
   Index: []},
  'efficiency_scale': 0.38,
  'cardinality_detail': [('STATION', 0.0